# files

> Files and cells over the gateway's contents and cells APIs

In [ ]:
#| default_exp files

[Rustygate](https://github.com/AnswerDotAI/rustygate) provides a Jupyter-style `/api/contents` API for files and directories and `/api/cells` for editing cells inside notebooks.

Use `JupyAsyncFilesClient` to address files by path. `JupyAsyncCellsClient` works with one notebook's cells. A kernel client bound to that notebook receives `cell_ops` change messages on its `cells` channel. `apply_ops` updates a local list of cells from those messages. Attach `JmsgQueues` if you want to read the broadcasts with `get_jmsg`.

In [ ]:
#| export
import json
from base64 import b64encode, b64decode
from fastcore.basics import patch, patch_to
from fasttransport.errors import APIError
from jupyasyncclient.core import KernelApi

In [ ]:
import asyncio, tempfile
from pathlib import Path
from queue import Empty
from fastcore.test import test_eq, expect_fail
from rustygate.tools import start_gateway
from jupyasyncclient import JupyAsyncKernelClient, JmsgQueues

## The files client

Both clients inherit `KernelApi`'s base URL, authentication, and HTTP transport. You can supply an HTTP client or let the transport create one for each request.

Use `session_id` to identify the author of a change. Rustygate suppresses broadcasts to websockets with that same id. With the default `None`, writes have no author and reach all subscribers.

For conditional file writes, pass `expected_hash`. A stale hash raises `HashMismatch`, whose `hash` contains the server's current hash.

In [ ]:
#| export
class HashMismatch(Exception):
    "A conditional write failed; `hash` is the server's current file hash."
    def __init__(self, hash):
        super().__init__(f'expected_hash is stale; the current hash is {hash}')
        self.hash = hash

class JupyAsyncFilesClient(KernelApi):
    "Files and directories over the gateway's contents API."
    def __init__(self, base_url, token=None, session_id=None, headers=None, timeout=30, http_client=None, verify=True):
        super().__init__(base_url, token=token, headers=headers, timeout=timeout, http_client=http_client, verify=verify)
        self.session_id = session_id

`_op` calls an operation from the bundled OpenAPI specification. It adds this client's `session_id` and omits arguments with value `None`. Operations that have no `session_id` parameter ignore it. An HTTP 409 response containing a hash becomes `HashMismatch`.

The generated operation separates query parameters from body fields. For example, `get(path)` requests a model and `put(path, type='directory')` requests directory creation. `patch` passes its body through `body_` because a rename has two paths: the URL's source path and the body's destination `path`.


In [ ]:
#| export
@patch
async def _op(self:JupyAsyncFilesClient, op, **kw):
    "Call spec op `op` with `session_id` attached and None values dropped; a conditional-write 409 raises `HashMismatch`."
    kw = {k:v for k,v in dict(kw, session_id=self.session_id).items() if v is not None}
    try: return await op(**kw)
    except APIError as e:
        if e.status_code==409 and isinstance(e.raw, dict) and 'hash' in e.raw: raise HashMismatch(e.raw['hash']) from e
        raise


Use `get` to read a contents model, `put` to write, and `post` to copy through the contents API:

In [ ]:
#| export
@patch
async def get(self:JupyAsyncFilesClient, path='', **kwargs):
    "The model at `path`; `kwargs` become query parameters, e.g. `fields`."
    if not path: return await self._op(self.api.contents.get_root, **kwargs)
    return await self._op(self.api.contents.get_path, path=path, **kwargs)

@patch
async def put(self:JupyAsyncFilesClient, path, expected_hash=None, **kwargs):
    "PUT to the contents API; `kwargs` are its body and query fields."
    return await self._op(self.api.contents.put_path, path=path, expected_hash=expected_hash, **kwargs)

@patch
async def post(self:JupyAsyncFilesClient, path, expected_hash=None, **kwargs):
    "POST to the contents API; `kwargs` are its body and query fields."
    return await self._op(self.api.contents.post_path, path=path, expected_hash=expected_hash, **kwargs)

For `patch`, pass the source path positionally. You can then use `path=` for the destination in the body. `delete` removes a file or an empty directory:

In [ ]:
#| export
@patch_to(JupyAsyncFilesClient)
async def patch(self, path, /, expected_hash=None, **kwargs):
    "PATCH with `kwargs` as the JSON body; `path` is positional-only, freeing the name for the body."
    return await self._op(self.api.contents.patch_path, path=path, expected_hash=expected_hash, body_=kwargs)

@patch
async def delete(self:JupyAsyncFilesClient, path, expected_hash=None):
    "Delete a file or an empty directory."
    return await self._op(self.api.contents.delete_path, path=path, expected_hash=expected_hash)

In [ ]:
#| export
@patch
async def write(self:JupyAsyncFilesClient, path, content, expected_hash=None, unique=False):
    "Write text or base64-encoded bytes and return the model with its new hash. With `unique`, choose a free `name_n.ext`."
    c,f = (b64encode(content).decode(),'base64') if isinstance(content, bytes) else (content,'text')
    return await self.put(path, expected_hash=expected_hash, unique=unique or None, content=c, format=f)

@patch
async def read(self:JupyAsyncFilesClient, path):
    "A file's contents: `str` for text, `bytes` for binary."
    m = await self.get(path, fields='content')
    return b64decode(m['content']) if m['format']=='base64' else m['content']

@patch
async def listing(self:JupyAsyncFilesClient, path='', fields=None):
    "The entries of directory `path`; `fields='hash'` adds each file's hash."
    return (await self.get(path, fields=fields))['content']


Let's start a local rustygate process with a temporary directory as its files root. All the following writes stay in that directory:

In [ ]:
root = Path(tempfile.mkdtemp())
g = start_gateway(('rustygate', '--root', root))
fc = JupyAsyncFilesClient(g.url)
m = await fc.write('notes.txt', 'hello')
m

```python
{ 'hash': '2cf24dba5fb0a30e26e83b2ac5b9e29e1b161e5c1fa7425e73043362938b9824',
  'mtime': 1788943661.5810122,
  'name': 'notes.txt',
  'path': 'notes.txt',
  'size': 5,
  'type': 'file',
  'writable': True}
```

`write` returns the new file hash. A directory listing with `fields='hash'` returns the same hash for that file:

In [ ]:
test_eq(await fc.read('notes.txt'), 'hello')
entry = next(e for e in await fc.listing(fields='hash') if e['name']=='notes.txt')
test_eq(entry['hash'], m['hash'])
await fc.get('notes.txt')

```python
{ 'mtime': 1788943661.5810122,
  'name': 'notes.txt',
  'path': 'notes.txt',
  'size': 5,
  'type': 'file',
  'writable': True}
```

If another write has changed the file, the old `expected_hash` no longer matches. The exception includes the current hash, so this example can retry without fetching the hash separately:

In [ ]:
try: await fc.write('notes.txt', 'clobber', expected_hash='0'*64)
except HashMismatch as e: cur = e.hash
test_eq(cur, m['hash'])
m2 = await fc.write('notes.txt', 'hello2', expected_hash=cur)
assert m2['hash'] != cur
await fc.read('notes.txt')

'hello2'

Pass `expected_hash=''` to require a new file. An existing path raises `HashMismatch` and keeps its content:

In [ ]:
await fc.write('fresh.txt', 'made', expected_hash='')
try: await fc.write('fresh.txt', 'again', expected_hash='')
except HashMismatch as e: cur = e.hash
test_eq(await fc.read('fresh.txt'), 'made')
cur

'ea0890697a77af0a2e054cccec587c8a42feb5cf38e778c6c6e2a96bfb945c0b'

`write` base64-encodes bytes for transport. `read` returns text when the stored bytes are valid UTF-8, otherwise bytes. Here the binary data round-trips unchanged:

In [ ]:
raw = bytes(range(256))
await fc.write('blob.bin', raw)
back = await fc.read('blob.bin')
test_eq(back, raw)
len(back)

256

`mkdir`, `rename`, and `copy` wrap the contents methods. Use `parents=True` to create missing directory ancestors.

A rename does not overwrite an existing destination. Rustygate returns HTTP 409 with that destination's hash, which the client converts to `HashMismatch`. To replace a destination, you must delete it explicitly first:


In [ ]:
#| export
@patch
async def mkdir(self:JupyAsyncFilesClient, path, parents=False):
    "Create directory `path`; `parents` creates missing ancestors like `mkdir -p`."
    return await self.put(path, parents=parents or None, type='directory')

@patch
async def rename(self:JupyAsyncFilesClient, path, to):
    "Rename and return the new model. An existing destination raises HashMismatch; it is never overwritten."
    return await self.patch(path, path=to)

@patch
async def copy(self:JupyAsyncFilesClient, src, to, unique=False):
    "Copy and return the new model. With `unique`, choose a free `name_n.ext`."
    return await self.post(to, unique=unique or None, copy_from=src)

In [ ]:
await fc.mkdir('sub')
await fc.mkdir('deep/a/b', parents=True)
await fc.copy('notes.txt', 'sub/notes.txt')
await fc.rename('sub/notes.txt', 'sub/renamed.txt')
with expect_fail(HashMismatch): await fc.rename('sub/renamed.txt', 'notes.txt')
test_eq([e['name'] for e in await fc.listing('sub')], ['renamed.txt'])
await fc.delete('sub/renamed.txt')
await fc.delete('sub')
await fc.delete('blob.bin')
[e['name'] for e in await fc.listing()]

['deep', 'fresh.txt', 'notes.txt']

`search` returns `/api/search` results as `{'paths', 'complete'}`. It can search file names or content. With `up`, it searches ancestors for the nearest matching entry, such as `.git`, a project file, or a per-directory configuration file:

In [ ]:
#| export
@patch
async def search(self:JupyAsyncFilesClient, path='', **kwargs):
    "The `/api/search` result for directory `path`: `kwargs` are its query parameters (`q`, `up`, `glob`, ...), documented in rustygate's DEV.md"
    return await self._op(self.api.search.search, path=path or None, **kwargs)

In [ ]:
test_eq((await fc.search(path='deep/a/b', up='notes.txt'))['paths'], ['notes.txt'])
await fc.search(glob='*.txt')


```python
{'complete': True, 'paths': ['fresh.txt', 'notes.txt']}
```

Pass `unique=True` to write or copy under the first free `name_n.ext` instead of overwriting. The response contains the final name and path:

In [ ]:
m3 = await fc.write('notes.txt', 'variant', unique=True)
test_eq(m3['name'], 'notes_1.txt')
m4 = await fc.copy('notes.txt', 'notes.txt', unique=True)
test_eq(m4['name'], 'notes_2.txt')
test_eq(await fc.read('notes_2.txt'), await fc.read('notes.txt'))
m4['path']

'notes_2.txt'

Use `edit` to change a JSON file, including notebook-level metadata that cell operations cannot address. It reads the content and hash, calls `f` on the parsed object, and writes the result with that hash as a condition.

`f` must modify the object in place. If a concurrent write wins, `edit` reads again and calls `f` on the new content. It makes up to three attempts by default, then raises `RuntimeError`. Other errors propagate without a retry.

In [ ]:
#| export
@patch
async def edit(self:JupyAsyncFilesClient, path, f, tries=3):
    "Read the JSON file at `path`, apply `f` to the parsed object, and write it back conditionally, retrying a lost race"
    for _ in range(tries):
        m = await self.get(path, fields='content,hash')
        o = json.loads(m['content'])
        f(o)
        try: return await self.put(path, expected_hash=m['hash'], content=json.dumps(o, sort_keys=True, indent=1), format='text')
        except HashMismatch: pass
    raise RuntimeError(f'edit kept losing races for {path}')

In [ ]:
await fc.write('cfg.json', json.dumps(dict(a=1)))
def _bump(o): o['a'] += 1
await fc.edit('cfg.json', _bump)
test_eq(json.loads(await fc.read('cfg.json'))['a'], 2)

## The cells client

Construct `JupyAsyncCellsClient` with the notebook path once. Its cell methods use that path. Inherited file methods still take explicit paths, which is useful for a dialog's sibling assets.

`cells`, `hashes`, and `apply` update the client's `hash` from their responses. You can compare that value with a change broadcast's file hash. `await client[id]` returns one cell; a tuple of ids returns a list.

In [ ]:
#| export
class JupyAsyncCellsClient(JupyAsyncFilesClient):
    "One notebook's cells over the gateway's cells API."
    def __init__(self, base_url, path, token=None, session_id=None, headers=None, timeout=30, http_client=None, verify=True):
        super().__init__(base_url, token=token, session_id=session_id, headers=headers, timeout=timeout, http_client=http_client, verify=verify)
        self.path,self.hash,self.matched = str(path),None,None

    def __getitem__(self, ids): return self._lookup(ids)

In [ ]:
#| export
def _cs(v):
    "A comma-separated str from a str, an int, an iterable, or None"
    if v is None or isinstance(v, str): return v
    return ','.join(map(str, v)) if hasattr(v, '__iter__') else str(v)

@patch
async def cells(self:JupyAsyncCellsClient,
    ids=None, # Cell ids to keep: comma-separated str, or a list
    idx=None, # Cell positions to keep: comma-separated str, or a list of ints, 0-based, negative from the end; unions with `ids`
    q=None, # Keep only cells whose source matches this regex (multiline, smart-case)
    cell_type=None, # Keep only cells of this type: 'code', 'markdown', or 'raw'
    meta=None, # Keep only cells whose metadata contains this dict as a recursive subset; a None value means "key present"
    meta_not=None, # Drop cells whose metadata contains this dict as a recursive subset
    limit=None, # Keep at most this many cells after filtering
    context=None, # Also return this many neighbours either side of each kept cell; `self.matched` then names the true matches
    fields=None, # Comma-separated extras: 'hashes', 'meta', 'attachments'
):
    "The notebook's cells in document order, optionally selected and filtered (the gateway's cells GET stages)."
    m = await self._op(self.api.cells.get_cells, path=self.path, ids=_cs(ids), idx=_cs(idx), q=q, cell_type=cell_type,
        meta=None if meta is None else json.dumps(meta), meta_not=None if meta_not is None else json.dumps(meta_not),
        limit=limit, context=context, fields=fields)
    self.hash = m['hash']
    self.matched = m.get('matched')
    return m['cells']


Use `hashes` to fetch `{'id', 'hash'}` rows without full cell content. `apply` submits an atomic batch and returns the added ids. `_lookup` implements the bracket notation:

In [ ]:
#| export
@patch
async def hashes(self:JupyAsyncCellsClient):
    "Per-cell `{'id','hash'}` rows: the cheap form for sync."
    m = await self._op(self.api.cells.get_cells, path=self.path, fields='hashes')
    self.hash = m['hash']
    return m['cells']

@patch
async def apply(self:JupyAsyncCellsClient, ops):
    "Apply `ops` atomically, returning ids of added cells."
    m = await self._op(self.api.cells.post_cells, path=self.path, ops=ops)
    self.hash = m['hash']
    return m['added_ids']

@patch
async def _lookup(self:JupyAsyncCellsClient, ids):
    one = isinstance(ids, str)
    want = [ids] if one else list(ids)
    got = await self.cells(ids=want)
    if len(got)!=len(want): raise KeyError(', '.join(i for i in want if i not in {c['id'] for c in got}))
    return got[0] if one else got

We'll create a notebook through the files client. Any nbformat producer can write the file. For an unbound notebook, the cells API reads it from disk. For a notebook bound to a kernel, the gateway serves its held state:

In [ ]:
cells = [dict(id='aaa1', cell_type='code', source='1+1', metadata={}, outputs=[], execution_count=None),
    dict(id='bbb2', cell_type='markdown', source='# hi', metadata={})]
await fc.write('d.ipynb', json.dumps(dict(nbformat=4, nbformat_minor=5, metadata={}, cells=cells)))
nb = JupyAsyncCellsClient(g.url, 'd.ipynb')
[c['id'] for c in await nb.cells()]

['aaa1', 'bbb2']

Bracket lookup works like fastlite's id lookup, with `await` for the request. A missing id raises `KeyError`. Multiple cells come back in document order:

In [ ]:
c = await nb['bbb2']
test_eq(c['source'], '# hi')
pair = await nb['aaa1','bbb2']
test_eq([c['id'] for c in pair], ['aaa1','bbb2'])
try: await nb['nope']
except KeyError as e: err = str(e)
err

"'nope'"

The server applies a cell-operation batch atomically. It fills in omitted defaults for a sparse `add` and returns added ids in operation order. An `update` replaces the fields you supply rather than merging their contents. If an operation fails, the server leaves the notebook unchanged:

In [ ]:
added = await nb.apply([
    dict(op='add', cell=dict(cell_type='code', source='2+2'), after='aaa1'),
    dict(op='update', id='bbb2', source='# hello'),
])
new_id, = added
[c['id'] for c in await nb.cells()]

['aaa1', '1eac427f', 'bbb2']

An HTTP 409 without a hash remains an `APIError`. For example, the cells API cannot edit an unbound file that doesn't parse as a notebook. Deleting a kernel-bound path also returns 409: shut down the kernel before deleting its file.

In [ ]:
await fc.write('junk.ipynb', '{not json')
bad = JupyAsyncCellsClient(g.url, 'junk.ipynb')
try: await bad.apply([dict(op='update', id='aaa1', source='x')])
except APIError as e: code = e.status_code
test_eq(code, 409)

## Applying ops

`apply_ops` updates a list of cell dictionaries in place from broadcast operations. A client can keep its local view current without fetching the notebook after each change.

The gateway converts richer request operations, such as metadata `merge`, into add/update/delete broadcasts containing their results. The local helper applies those cell operations in order. Handle a file-level `rename` yourself by updating the client's path. Passing a rename to this helper raises `ValueError`.

In [ ]:
#| export
def _update_cell(cell, fields):
    cell.update({k:v for k,v in fields.items() if k not in ('op','id')})
    if cell['cell_type']=='code':
        cell.setdefault('outputs', [])
        cell.setdefault('execution_count', None)
    else:
        cell.pop('outputs', None)
        cell.pop('execution_count', None)

def apply_ops(cells, ops):
    "Apply cell broadcast operations in order to `cells` in place and return that list."
    for o in ops:
        ids = [c['id'] for c in cells]
        op = o['op']
        if op=='update' and o['id'] in ids: _update_cell(cells[ids.index(o['id'])], o)
        elif op in ('add','update'):
            c = dict(o['cell']) if op=='add' else {k:v for k,v in o.items() if k!='op'}
            if c.get('id') in ids: _update_cell(cells[ids.index(c['id'])], c)
            else:
                c.setdefault('cell_type', 'code')
                c.setdefault('metadata', {})
                if c['cell_type']=='code':
                    c.setdefault('outputs', [])
                    c.setdefault('execution_count', None)
                anchor = o.get('after') or o.get('before')
                at = ids.index(anchor) + bool(o.get('after')) if anchor in ids else len(cells)
                cells.insert(at, c)
        elif op=='delete':
            if o['id'] in ids: del cells[ids.index(o['id'])]
        else: raise ValueError(f"unhandled op: {op}")
    return cells

Here we add a code cell, delete another cell, and turn the new cell into Markdown. The type change removes its execution count and outputs. The server and local view agree on the cell contents and order:

In [ ]:
view = await nb.cells()
ops = [dict(op='add', cell=dict(id='ccc3', cell_type='code', source='3', metadata={}, outputs=[], execution_count=None), before='aaa1'),
    dict(op='delete', id='bbb2'), dict(op='update', id='ccc3', cell_type='markdown', source='# Calculation')]
await nb.apply(ops)
apply_ops(view, ops)
test_eq(view, await nb.cells())
[(c['id'], c['cell_type']) for c in view]

['ccc3', 'aaa1', '1eac427f']

Cell operations tolerate missing and duplicate ids in these cases:

- Updating a missing id adds the cell at the end. Defaults include `cell_type='code'`, empty metadata, and, for code cells, empty outputs and a null execution count.
- Adding an existing id updates that cell in place and ignores the anchor.
- Adding relative to a missing anchor appends the cell.
- Deleting a missing id has no effect.

These rules do not make malformed operations valid. Here the duplicate add turns `ccc3` back into code. Its outputs start empty and its execution count is null. The local result matches the server:

In [ ]:
ops = [dict(op='update', id='zzz9', source='9'), dict(op='delete', id='zzz8'),
    dict(op='add', cell=dict(id='ccc3', cell_type='code', source='3b'), after='ggg7'), dict(op='add', cell=dict(id='ddd4', source='4'), after='ggg7')]
await nb.apply(ops)
apply_ops(view, ops)
srv = await nb.cells()
test_eq([c['id'] for c in view], [c['id'] for c in srv])
for i in ('zzz9', 'ccc3', 'ddd4'): test_eq(next(c for c in view if c['id']==i), next(c for c in srv if c['id']==i))


## Change broadcasts

Create a kernel with `path` to subscribe its websocket clients to that notebook's changes. Broadcasts use message type `cell_ops` on channel `cells`. Their content contains `path`, `hash`, and `ops`.

The gateway excludes the change's author by matching `session_id`. Our `nb` client has no session id, so its writes reach every subscriber:

In [ ]:
kc = JupyAsyncKernelClient(g.url)
await kc.start_kernel(path='d.ipynb')
qs = JmsgQueues(kc, queues=('jmsg',), merge=dict(iopub='jmsg', stdin='jmsg', cells='jmsg'))
kc.start_channels()
await kc.wait_for_ready(timeout=60)
kc.channels_running

True

In [ ]:
await nb.apply([dict(op='update', id='aaa1', source='40+2')])
m = await qs.jmsg_for('cell_ops', timeout=15)
test_eq(m['header']['msg_type'], 'cell_ops')
m['content']

{'path': 'd.ipynb',
 'hash': 'a88e732ed449dd96e8485c60c126019ab6d0716fd0f99979b39d4a49a5899228',
 'ops': [{'op': 'update',
   'id': 'aaa1',
   'cell_type': 'code',
   'execution_count': None,
   'metadata': {},
   'outputs': [],
   'source': '40+2'}]}

Apply the broadcast's operations to the local cells. Its `hash` identifies the notebook state after that change:

In [ ]:
apply_ops(view, m['content']['ops'])
test_eq(next(c['source'] for c in view if c['id']=='aaa1'), '40+2')
test_eq(m['content']['hash'], nb.hash)
m['content']['path']

'd.ipynb'

This cells client shares `kc`'s `session_id`. Its writes do not echo back through `kc`'s websocket:

In [ ]:
own = JupyAsyncCellsClient(g.url, 'd.ipynb', session_id=kc.session_id)
await own.apply([dict(op='update', id='aaa1', source='6*7')])
try:
    await qs.jmsg_for('cell_ops', timeout=1.5)
    heard = True
except Empty: heard = False
test_eq(heard, False)

To save execution output into the bound notebook, put the cell id in the request metadata as `cellId`. The gateway updates that cell's held state before forwarding its iopub messages. A cells fetch after the execution's `idle` can read the saved output.

An execution without `cellId`, or with an id absent from the notebook, still runs but does not persist its output:

In [ ]:
kc.execute('6*7', metadata=dict(cellId='aaa1'))
await qs.jmsg_for('status', pred=lambda m: m['content']['execution_state'] == 'idle', timeout=15)
c, = await nb.cells(ids=['aaa1'])
test_eq(c['outputs'][0]['data']['text/plain'], '42')
test_eq(c['execution_count'], 1)

The cells GET applies selection and filters in order. `ids` and `idx` select a union of cells in document order. Positions start at zero; negative positions count from the end. With neither selector, it starts with all cells.

The selected cells must pass `q`, `cell_type`, `meta`, and `meta_not` when supplied. `q` is a multiline, smart-case regex over source text. `limit` keeps the first matching cells, then `context` adds neighbours. When context is requested, `nb.matched` lists the matches before that expansion.

In [ ]:
test_eq([c['id'] for c in await nb.cells(idx=[0,-1])], ['ccc3', 'ddd4'])
test_eq([c['id'] for c in await nb.cells(ids='zzz9', idx=[0])], ['ccc3', 'zzz9'])
test_eq([c['id'] for c in await nb.cells(q=r'\*')], ['aaa1'])


For `meta`, provide a dictionary that the cell's metadata must contain as a recursive subset. `meta_not` excludes cells matching that subset. A `None` value means the key must exist, whatever its value.

A metadata `merge` uses `None` differently: it deletes that key. This example adds a tag, finds the tagged cell, then removes the tag:

In [ ]:
await nb.apply([dict(op='merge', id='zzz9', metadata=dict(tag=1))])
test_eq([c['id'] for c in await nb.cells(meta=dict(tag=None))], ['zzz9'])
await nb.apply([dict(op='merge', id='zzz9', metadata=dict(tag=None))])

[]

With `context=1`, the result includes up to one neighbour on either side of each match. `matched` identifies the cells that matched the query:

In [ ]:
got = await nb.cells(q=r'\*', context=1)
test_eq((len(got), nb.matched), (3, ['aaa1']))
[c['id'] for c in got]

['ccc3', 'aaa1', '1eac427f']

A `{'op':'rename','to':...}` broadcast changes the notebook path. Update the cells client's `path` before making further requests.

Broadcast adds and updates contain the resulting cell fields. They need no follow-up read for those fields. Attachment bytes travel separately: attachment events identify the affected attachment, and clients fetch its bytes when needed.

While a notebook is bound, the gateway keeps its current parsed state in memory. Foreign writes with duplicate ids produce repaired ids and ordinary change operations. A foreign write that cannot parse leaves the held state unchanged. Reads continue to serve that state. After a grace interval for non-atomic writers, the gateway restores it to disk. It also restores a bound file that disappears. These events do not reset the stream.

If a reconnect reports dropped messages, resynchronize instead of continuing from an incomplete stream. Call `hashes`, compare the per-cell hashes with the local view, and fetch changed cells with `ids`.


In [ ]:
#| hide
await kc.shutdown_kernel()
g.stop()

In [ ]:
#| hide
import nbdev
nbdev.nbdev_export()